# People Counter CLI tutorial

This notebook teaches reproducible CLI invocation from a terminal, Python application, scheduler, or notebook. Commands are built as argument lists and never use `shell=True`.

## 1. Install one hardware variant

From the repository root:

```bash
uv sync --extra cpu --extra experiments
uv run --extra cpu people-counter --help
```

For a compatible NVIDIA host, replace both occurrences of `cpu` with `gpu`. The command requires an explicit device and will not silently fall back.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the people-counter repository")


ROOT = find_repository_root(Path.cwd().resolve())
VIDEO = ROOT / "samples" / "three_people_walking.mp4"
OUTPUT_DIR = ROOT / "outputs" / "cli_tutorial"
RUN_INFERENCE = False

# This module form uses the same parser as the installed people-counter command.
CLI = [sys.executable, "-m", "people_counter.cli"]

if not VIDEO.is_file():
    raise FileNotFoundError(f"Sample video not found: {VIDEO}")

print(f"Repository: {ROOT}")
print(f"CLI executable: {shlex.join(CLI)}")
print(f"Inference enabled: {RUN_INFERENCE}")

## 2. Discover commands and options

The installed `people-counter` executable and `python -m people_counter.cli` expose the same unified parser. Use the module form inside notebooks to guarantee that the active kernel's environment is used.

In [ ]:
help_result = subprocess.run(
    [*CLI, "--help"],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(help_result.stdout)

In [ ]:
subcommand_help = subprocess.run(
    [*CLI, "rtdetr-osnet", "--help"],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(subcommand_help.stdout)

## 3. Build a safe RT-DETR/OSNet command

Arguments stay separate so paths and user-provided values cannot be interpreted as shell syntax. This example samples at 3 FPS, uses the CPU-safe batch size, sets the activation threshold, and writes timestamped CSV files to a dedicated directory.

To count directed crossings, replace the four sample line coordinates after checking the video's dimensions. Reversing the endpoints swaps `in` and `out`.

In [ ]:
# Replace with (x1, y1, x2, y2) after checking the source dimensions.
COUNTING_LINE = None

rtdetr_command = [
    *CLI,
    "rtdetr-osnet",
    str(VIDEO),
    "--device",
    "cpu",
    "--sample-fps",
    "3",
    "--batch-size",
    "1",
    "--detection-threshold",
    "0.6",
    "--output-dir",
    str(OUTPUT_DIR),
]

if COUNTING_LINE is not None:
    rtdetr_command.extend(
        ["--line", *(str(coordinate) for coordinate in COUNTING_LINE)]
    )

print(shlex.join(rtdetr_command))

## 4. Execute and surface failures

Set `RUN_INFERENCE = True` in the setup cell when ready. `check_returncode()` turns a nonzero CLI status into a notebook failure, which is important for schedulers and Microsoft Fabric pipeline activities.

In [ ]:
completed = None

if RUN_INFERENCE:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        rtdetr_command,
        cwd=ROOT,
        check=False,
        capture_output=True,
        text=True,
    )
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    completed.check_returncode()
else:
    print("Inference skipped. Set RUN_INFERENCE = True to execute the command.")

## 5. Discover and read timestamped outputs

The CLI writes identity telemetry for every run and writes a separate line-count CSV when `--line X1 Y1 X2 Y2` is supplied. Timestamped names prevent accidental overwrite.

In [ ]:
telemetry_files = (
    sorted(OUTPUT_DIR.glob("*_telemetry_*.csv"))
    if OUTPUT_DIR.is_dir()
    else []
)
line_count_files = (
    sorted(OUTPUT_DIR.glob("*_line_counts_*.csv"))
    if OUTPUT_DIR.is_dir()
    else []
)

print("Telemetry files:", [path.name for path in telemetry_files])
print("Line-count files:", [path.name for path in line_count_files])

In [ ]:
if not telemetry_files:
    print("No tutorial telemetry exists yet.")
else:
    import pandas as pd

    latest_telemetry = pd.read_csv(telemetry_files[-1])
    display(latest_telemetry.head())

    if line_count_files:
        latest_line_counts = pd.read_csv(line_count_files[-1])
        display(latest_line_counts.head())

## 6. Compare RF-DETR/BoT-SORT

The comparison pipeline accepts the same common sampling, confidence, device, line, and output options. Camera-motion compensation can be selected explicitly with `--cmc` or `--no-cmc`.

In [ ]:
botsort_command = [
    *CLI,
    "rfdetr-botsort",
    str(VIDEO),
    "--device",
    "cpu",
    "--sample-fps",
    "3",
    "--batch-size",
    "1",
    "--no-cmc",
    "--output-dir",
    str(OUTPUT_DIR),
]

print(shlex.join(botsort_command))

## 7. Legacy aliases

`people-counter-rtdetr` and `people-counter-rfdetr` remain available for compatibility. New automation should prefer the unified `people-counter rtdetr-osnet ...` and `people-counter rfdetr-botsort ...` forms.

## 8. Microsoft Fabric orchestration

For Fabric notebooks, the Python SDK is usually preferable because records can be written directly to Delta tables. If an existing pipeline standard requires the CLI:

1. Upload the built wheel and dependencies to a Fabric Environment.
2. Attach that environment to the notebook activity.
3. Resolve the OneLake video to a local path OpenCV can read.
4. Invoke the CLI with `subprocess.run([...], check=True)`.
5. Copy or ingest the generated CSVs into the Lakehouse.
6. Let nonzero return codes propagate so the activity can retry or alert.

Frames within one video are stateful and must remain sequential. Scale by assigning separate videos to separate activities.

## Next steps

- Use `--sample-fps all` only when every source frame is required.
- Increase GPU batch size gradually while monitoring memory.
- Store the full argument list with each run for reproducibility.
- Add a stable run ID and idempotent ingestion when an orchestrator can retry work.